In [2]:
# =============================================================
# [feature_engineering.ipynb - Cell 1] 입력 로드 & 구조 확인
# 입력: ../output/synthetic_anomaly_data.csv (이번 단계 주 입력)
#       ../reports/model_eda/normal_baseline_by_tool.csv (상대 Feature용)
# 이번 셀: 로드/검증만. Window 생성·Feature 계산 없음.
# =============================================================
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

SYNTH_PATH    = Path("../output/synthetic_anomaly_data.csv")
BASELINE_PATH = Path("../reports/model_eda/normal_baseline_by_tool.csv")

# --- 1) 로드 ---
synth = pd.read_csv(SYNTH_PATH, encoding="utf-8-sig", low_memory=False)
baseline = pd.read_csv(BASELINE_PATH, encoding="utf-8-sig")

# --- 2) timestamp dtype 확인/변환 (형식 자동 추론: ISO/쉼표 모두 대응) ---
synth["timestamp"] = pd.to_datetime(synth["timestamp"], errors="coerce")

print("[dtype 점검]")
print("timestamp dtype :", synth["timestamp"].dtype,
      "| 결측:", synth["timestamp"].isna().sum())
if "is_synthetic" in synth.columns:
    print("is_synthetic dtype:", synth["is_synthetic"].dtype,
          "| 예시값:", synth["is_synthetic"].iloc[0], type(synth["is_synthetic"].iloc[0]).__name__)

# --- 3) 필수 컬럼 존재 확인 ---
required = [
    "timestamp", "current_tool", "spindle_rpm", "spindle_rate", "spindle_load",
    "source_segment_id", "split", "anomaly_type", "stage", "label", "is_synthetic",
]
missing = [c for c in required if c not in synth.columns]
print("\n[필수 컬럼 누락]:", missing if missing else "없음 (모두 존재)")

base_required = [
    "current_tool", "insufficient_data",
    "load_median", "load_p95", "load_p99",
    "rpm_median", "rpm_error_median", "rpm_error_p95", "rpm_error_p99",
    "rpm_std_1min_p95", "load_std_1min_p95", "rpmerr_std_1min_p95",
]
base_missing = [c for c in base_required if c not in baseline.columns]
print("[baseline 필수 컬럼 누락]:", base_missing if base_missing else "없음 (모두 존재)")

# --- 4) shape / seed / split ---
print("\n" + "=" * 60)
print("[synthetic_anomaly_data.csv]")
print("shape:", synth.shape)
print("컬럼 수:", synth.shape[1])
print("seed 개수 (source_segment_id):", synth["source_segment_id"].nunique())
print("기간:", synth["timestamp"].min(), "~", synth["timestamp"].max())

print("\n[split 분포]")
print(synth["split"].value_counts().reindex(["train","validation","test"]).to_string())

# --- 5) label / stage / anomaly_type 분포 ---
print("\n[label 분포]  (0=normal, 1=warning+anomaly)")
print(synth["label"].value_counts().sort_index().to_string())

print("\n[stage 분포]")
print(synth["stage"].value_counts().reindex(["normal","warning","anomaly"]).to_string())

print("\n[anomaly_type 분포]")
print(synth["anomaly_type"].value_counts().to_string())

# stage ↔ label 정합성 (normal=0, warning/anomaly=1 이어야 함)
print("\n[stage × label 교차 (정합성 확인)]")
print(pd.crosstab(synth["stage"], synth["label"]))

# --- 6) NaN 확인 ---
print("\n[NaN 확인]")
na = synth.isna().sum()
na = na[na > 0]
print(na.to_string() if len(na) else "결측 없음 (모든 컬럼 NaN 0)")

# --- baseline 참고: 사용 가능 공구 ---
print("\n[baseline] insufficient_data=True 공구:",
      baseline.loc[baseline["insufficient_data"], "current_tool"].tolist())

print("\n[안내] 로드/검증 완료. Window 생성은 Cell 2에서 진행합니다.")

[dtype 점검]
timestamp dtype : datetime64[us] | 결측: 0
is_synthetic dtype: bool | 예시값: True bool

[필수 컬럼 누락]: 없음 (모두 존재)
[baseline 필수 컬럼 누락]: 없음 (모두 존재)

[synthetic_anomaly_data.csv]
shape: (48240, 38)
컬럼 수: 38
seed 개수 (source_segment_id): 462
기간: 2026-05-01 00:00:04.522000 ~ 2026-05-30 18:51:38.211000

[split 분포]
split
train         37977
validation     4458
test           5805

[label 분포]  (0=normal, 1=warning+anomaly)
label
0    14647
1    33593

[stage 분포]
stage
normal     14647
warning    14392
anomaly    19201

[anomaly_type 분포]
anomaly_type
normal             14647
load_rise          11267
rpm_instability    11195
rpm_error_drift    11131

[stage × label 교차 (정합성 확인)]
label        0      1
stage                
anomaly      0  19201
normal   14647      0
warning      0  14392

[NaN 확인]
error_message       48240
interval_seconds        1

[baseline] insufficient_data=True 공구: [9, 7, 20]

[안내] 로드/검증 완료. Window 생성은 Cell 2에서 진행합니다.


In [3]:
# =============================================================
# [feature_engineering.ipynb - Cell 2] 1분 Window 분할 & 구조 검증
# 규칙: seed_start 기준 경과초 → window_id = floor(elapsed/60)
#       seed(source_segment_id) 내부에서만, split 경계 안 넘음(같은 seed=한 split)
#       label = window 내부 label.max(), stage = 대표(최빈) stage
# 이번 셀: Window 인덱스 테이블 + 구조 검증만. Feature 계산 없음.
# =============================================================

# seed별 시작시각(seed_start) 확보: 각 source_segment_id의 최소 timestamp
seed_start = (synth.groupby("source_segment_id")["timestamp"]
                   .min().rename("seed_start"))
d = synth.merge(seed_start, on="source_segment_id", how="left")

# --- 경과초 & window_id = floor(elapsed / 60) ---
d["elapsed_s"] = (d["timestamp"] - d["seed_start"]).dt.total_seconds()
d["window_id"] = np.floor(d["elapsed_s"] / 60).astype(int)

# --- Window 단위 집계 (seed × window_id 로 그룹, seed는 한 split에만 속하므로 안전) ---
def rep_stage(s):
    return s.mode().iat[0]   # 대표(최빈) stage

win = (d.groupby(["source_segment_id", "window_id"])
         .agg(window_start=("timestamp", "min"),
              window_end=("timestamp", "max"),
              win_n=("timestamp", "size"),
              current_tool=("current_tool", "first"),
              split=("split", "first"),
              anomaly_type=("anomaly_type", "first"),
              stage=("stage", rep_stage),
              label=("label", "max"),
              n_stage=("stage", "nunique"))   # 한 Window에 섞인 stage 종류 수
         .reset_index())

# 다음 셀에서 사용할 수 있게 elapsed/window_id가 붙은 행 테이블도 보관
rows_with_window = d
window_table = win.copy()

# ---------------- 검증 출력 ----------------
print("=" * 60)
print(f"총 Window 개수: {len(win):,}")
print("=" * 60)

print("\n[seed당 Window 개수 분포]")
per_seed = win.groupby("source_segment_id").size()
print(per_seed.describe()[["min","50%","max"]].rename({"50%":"median"}).to_string()
      if False else per_seed.describe().round(2).to_string())
print("seed당 Window 개수 value_counts:")
print(per_seed.value_counts().sort_index().to_string())

print("\n[win_n (Window당 행 수) 요약]")
print(f"min={win['win_n'].min()}  median={win['win_n'].median()}  max={win['win_n'].max()}")

print("\n[stage별 Window 개수]")
print(win["stage"].value_counts().reindex(["normal","warning","anomaly"]).to_string())

print("\n[label별 Window 개수]")
print(win["label"].value_counts().sort_index().to_string())

print("\n[split별 Window 개수]")
print(win["split"].value_counts().reindex(["train","validation","test"]).to_string())

# --- 검증 1: 한 Window가 두 stage를 섞어 포함했는지 ---
mixed = win[win["n_stage"] > 1]
print(f"\n[검증] stage가 섞인 Window 개수: {len(mixed)}  (0이어야 정상)")
if len(mixed):
    print(mixed[["source_segment_id","window_id","stage","n_stage"]].head().to_string(index=False))

# --- 검증 2: window_id가 0~9 범위를 벗어났는지 ---
out_of_range = win[(win["window_id"] < 0) | (win["window_id"] > 9)]
print(f"[검증] window_id 0~9 벗어난 Window 개수: {len(out_of_range)}  (0이어야 정상)")
if len(out_of_range):
    print(out_of_range["window_id"].value_counts().sort_index().to_string())

# --- 검증 3: window_id ↔ stage/label 매핑이 설계대로인지 ---
print("\n[검증] window_id별 stage/label 분포 (설계: 0~2 normal/0, 3~5 warning/1, 6~9 anomaly/1)")
print(pd.crosstab(win["window_id"], [win["stage"], win["label"]]))

print("\n[안내] Window 분할·검증 완료. Feature 계산은 Cell 3에서 진행합니다.")

총 Window 개수: 4,620

[seed당 Window 개수 분포]
count    462.0
mean      10.0
std        0.0
min       10.0
25%       10.0
50%       10.0
75%       10.0
max       10.0
seed당 Window 개수 value_counts:
10    462

[win_n (Window당 행 수) 요약]
min=9  median=10.0  max=11

[stage별 Window 개수]
stage
normal     1386
warning    1386
anomaly    1848

[label별 Window 개수]
label
0    1386
1    3234

[split별 Window 개수]
split
train         3630
validation     430
test           560

[검증] stage가 섞인 Window 개수: 0  (0이어야 정상)
[검증] window_id 0~9 벗어난 Window 개수: 0  (0이어야 정상)

[검증] window_id별 stage/label 분포 (설계: 0~2 normal/0, 3~5 warning/1, 6~9 anomaly/1)
stage     anomaly normal warning
label           1      0       1
window_id                       
0               0    462       0
1               0    462       0
2               0    462       0
3               0      0     462
4               0      0     462
5               0      0     462
6             462      0       0
7             462      0       0
8           

In [4]:
# =============================================================
# [feature_engineering.ipynb - Cell 3] 1분 Window Feature Engineering
# - 기본 Feature: RPM/LOAD/RPM_ERROR 집계
# - 상대 기준(baseline)은 split==train & stage==normal Window에서만 계산 → 전체 적용
# - normal_baseline_by_tool.csv는 상대 기준으로 사용하지 않음 (누수 방지)
# 입력: rows_with_window (Cell 2), window_table (Cell 2)
# 주의: XGBoost 학습/StandardScaler 없음
# =============================================================

# --- 0) 행 단위 rpm_error 생성 (저장파일엔 없으므로 여기서 계산) ---
r = rows_with_window.copy()
r["rpm_error"] = r["spindle_rate"] - r["spindle_rpm"]

# --- 1) Window별 기본 Feature 집계 ---
g = r.groupby(["source_segment_id", "window_id"])
feat = g.agg(
    # RPM
    rpm_mean=("spindle_rpm", "mean"),
    rpm_std=("spindle_rpm", "std"),
    rpm_min=("spindle_rpm", "min"),
    rpm_max=("spindle_rpm", "max"),
    # LOAD
    load_mean=("spindle_load", "mean"),
    load_std=("spindle_load", "std"),
    load_max=("spindle_load", "max"),
    # RPM_ERROR
    rpm_error_mean=("rpm_error", "mean"),
    rpm_error_std=("rpm_error", "std"),
    rpm_error_absmax=("rpm_error", lambda s: s.abs().max()),
).reset_index()

feat["rpm_range"] = feat["rpm_max"] - feat["rpm_min"]

# std는 win_n=1일 때 NaN 가능 → 0으로 (실제로는 win_n>=9라 발생 안 하지만 안전차원)
for c in ["rpm_std", "load_std", "rpm_error_std"]:
    feat[c] = feat[c].fillna(0.0)

# --- 2) Window 메타(추적정보) 병합 ---
meta_cols = ["source_segment_id", "window_id", "window_start", "window_end",
             "win_n", "current_tool", "split", "anomaly_type", "stage", "label"]
feat = window_table[meta_cols].merge(feat, on=["source_segment_id", "window_id"])

# =============================================================
# 3) Train + normal Window에서만 공구별 정상 기준 계산
# =============================================================
train_normal = feat[(feat["split"] == "train") & (feat["stage"] == "normal")]

# 공구별 정상 기준:
#   load_center      = 정상 load_mean 의 중앙값
#   rpm_std_center   = 정상 rpm_std 의 중앙값 (변동성 기준)
#   rpmerr_center    = 정상 rpm_error_mean 의 중앙값 (편차 중심)
#   rpmerr_tol       = 정상 rpm_error_mean 의 허용 편차 (P95-중심, 최소 하한 보장)
tool_ref = train_normal.groupby("current_tool").agg(
    load_center=("load_mean", "median"),
    rpm_std_center=("rpm_std", "median"),
    rpmerr_center=("rpm_error_mean", "median"),
    rpmerr_p95=("rpm_error_mean", lambda s: s.quantile(0.95)),
    rpmerr_p05=("rpm_error_mean", lambda s: s.quantile(0.05)),
).reset_index()

# 허용 편차 = 중심에서 P95/P05까지 거리 중 큰 값 (양방향 이탈 대비)
tool_ref["rpmerr_tol"] = np.maximum(
    (tool_ref["rpmerr_p95"] - tool_ref["rpmerr_center"]).abs(),
    (tool_ref["rpmerr_center"] - tool_ref["rpmerr_p05"]).abs(),
)

# --- 안전 하한 (0으로 나누기/과소 baseline 방지). 신호 약화 막기 위해 작게 설정 ---
EPS_LOAD    = 1.0    # load_center 하한 (부하 최소 1%)
EPS_RPMSTD  = 1.0    # 정상 rpm_std 하한 (rpm) — 0인 공구 대비, 과대하지 않게 1
EPS_RPMERR  = 1.0    # rpm_error 허용편차 하한 (rpm)

tool_ref["load_center_safe"]  = tool_ref["load_center"].clip(lower=EPS_LOAD)
tool_ref["rpm_std_center_safe"] = tool_ref["rpm_std_center"].clip(lower=EPS_RPMSTD)
tool_ref["rpmerr_tol_safe"]   = tool_ref["rpmerr_tol"].clip(lower=EPS_RPMERR)

print("[Train+normal 기준으로 계산된 공구별 정상 기준] (상위 8개 공구)")
print(tool_ref[["current_tool","load_center","rpm_std_center",
                "rpmerr_center","rpmerr_tol"]].round(2).head(8).to_string(index=False))
print("\n공구 수:", tool_ref["current_tool"].nunique(),
      "| train+normal Window 수:", len(train_normal))

# =============================================================
# 4) 상대 Feature 생성 (train 기준을 전체 train/val/test에 적용)
# =============================================================
feat = feat.merge(
    tool_ref[["current_tool", "load_center_safe",
              "rpm_std_center_safe", "rpmerr_center", "rpmerr_tol_safe"]],
    on="current_tool", how="left",
)

# 1) load_vs_normal: 현재 load_mean 이 공구 정상 부하 대비 몇 배
feat["load_vs_normal"] = feat["load_mean"] / feat["load_center_safe"]

# 2) rpm_std_vs_normal: 현재 rpm_std 가 공구 정상 변동성 대비 몇 배
feat["rpm_std_vs_normal"] = feat["rpm_std"] / feat["rpm_std_center_safe"]

# 3) rpm_error_vs_normal: 공구 정상 편차 중심에서 벗어난 크기 / 허용 편차 (절댓값)
feat["rpm_error_vs_normal"] = (
    (feat["rpm_error_mean"] - feat["rpmerr_center"]).abs() / feat["rpmerr_tol_safe"]
)

# 사용한 임시 기준 컬럼 제거 (Feature만 남김)
feat = feat.drop(columns=["load_center_safe", "rpm_std_center_safe",
                          "rpmerr_center", "rpmerr_tol_safe"])

# 다음 셀(저장)에서 사용
feature_dataset = feat.copy()

# ---------------- 출력 ----------------
FEATURE_COLS = ["rpm_mean","rpm_std","rpm_min","rpm_max","rpm_range",
                "load_mean","load_std","load_max",
                "rpm_error_mean","rpm_error_std","rpm_error_absmax",
                "load_vs_normal","rpm_std_vs_normal","rpm_error_vs_normal"]

print("\n" + "=" * 60)
print("Window Feature 데이터 shape:", feat.shape)
print("=" * 60)
print("\n[Feature 컬럼 목록]")
print(FEATURE_COLS)

# NaN / inf 확인
nan_cnt = feat[FEATURE_COLS].isna().sum().sum()
inf_cnt = np.isinf(feat[FEATURE_COLS].to_numpy(dtype="float64")).sum()
print(f"\nNaN 개수: {nan_cnt} | inf 개수: {inf_cnt}")
if nan_cnt:
    print(feat[FEATURE_COLS].isna().sum()[lambda s: s>0].to_string())

print("\n[label 분포]")
print(feat["label"].value_counts().sort_index().to_string())
print("\n[split 분포]")
print(feat["split"].value_counts().reindex(["train","validation","test"]).to_string())

print("\n[상대 Feature 3종 통계 (split별 mean)]")
print(feat.groupby("split")[["load_vs_normal","rpm_std_vs_normal","rpm_error_vs_normal"]]
        .mean().round(3).reindex(["train","validation","test"]).to_string())
print("\n[상대 Feature 3종 describe]")
print(feat[["load_vs_normal","rpm_std_vs_normal","rpm_error_vs_normal"]]
        .describe(percentiles=[.5,.95]).round(3).to_string())

print("\n[안내] Feature 계산 완료. 저장은 다음 셀에서 진행합니다.")

[Train+normal 기준으로 계산된 공구별 정상 기준] (상위 8개 공구)
 current_tool  load_center  rpm_std_center  rpmerr_center  rpmerr_tol
            1         7.91             0.0           4.09        1.73
            2         5.73             0.0           2.00        0.16
            3        10.80             0.0           1.09        0.27
            4        10.64             0.0           1.09        0.27
            5        11.43             0.0          -3.90        0.20
            6        12.24             0.0           1.27        0.18
            8        12.60             0.0          -0.80        0.30
           10        13.30             0.0           1.36        0.36

공구 수: 14 | train+normal Window 수: 1089

Window Feature 데이터 shape: (4620, 24)

[Feature 컬럼 목록]
['rpm_mean', 'rpm_std', 'rpm_min', 'rpm_max', 'rpm_range', 'load_mean', 'load_std', 'load_max', 'rpm_error_mean', 'rpm_error_std', 'rpm_error_absmax', 'load_vs_normal', 'rpm_std_vs_normal', 'rpm_error_vs_normal']

NaN 개수: 180 | in

In [5]:
# NaN이 난 Window의 공구 = train+normal에 기준이 없는 공구
missing_tools = sorted(
    feature_dataset.loc[feature_dataset["load_vs_normal"].isna(), "current_tool"].unique()
)
print("train+normal 기준이 없어 NaN이 된 공구:", missing_tools)
print(feature_dataset[feature_dataset["load_vs_normal"].isna()]
      .groupby(["current_tool","split"]).size())

train+normal 기준이 없어 NaN이 된 공구: [np.int64(15), np.int64(16)]
current_tool  split     
15            validation    30
16            validation    30
dtype: int64


In [6]:
# =============================================================
# [feature_engineering.ipynb - Cell 4] NaN 제외 & training_dataset.csv 저장
# - 상대 Feature NaN Window(공구15·16, Train baseline 부재) 제외
# - 지정 컬럼만 저장 (스케일링 없음), 재로드 검증
# 입력: feature_dataset (Cell 3)
# =============================================================
from pathlib import Path

REL_FEATURES = ["load_vs_normal", "rpm_std_vs_normal", "rpm_error_vs_normal"]
OUT_PATH = Path("../output/training_dataset.csv")

df_feat = feature_dataset.copy()
n_before = len(df_feat)

# --- 1) 상대 Feature 중 하나라도 NaN인 Window 식별 ---
nan_mask = df_feat[REL_FEATURES].isna().any(axis=1)
excluded = df_feat[nan_mask]

# --- 2) 제외 내역 출력 ---
print("=" * 60)
print(f"[제외 대상] 상대 Feature NaN Window: {int(nan_mask.sum())}개")
print("=" * 60)
print("사유: Train 기간에 해당 공구의 정상(stage=normal) Window가 없어")
print("      공구별 정상 baseline을 만들 수 없음 → 상대 Feature 계산 불가")
print("\n[제외 Window: 공구 × split 별 개수]")
print(excluded.groupby(["current_tool", "split"]).size().to_string())

# --- 3) 제외 후 데이터 ---
df_final = df_feat[~nan_mask].reset_index(drop=True)
print(f"\n제외 전 Window: {n_before:,}  →  제외 후 Window: {len(df_final):,}")

# --- 4) NaN / inf 최종 검증 ---
FEATURE_COLS = ["rpm_mean","rpm_std","rpm_min","rpm_max","rpm_range",
                "load_mean","load_std","load_max",
                "rpm_error_mean","rpm_error_std","rpm_error_absmax",
                "load_vs_normal","rpm_std_vs_normal","rpm_error_vs_normal"]
nan_cnt = df_final[FEATURE_COLS].isna().sum().sum()
inf_cnt = np.isinf(df_final[FEATURE_COLS].to_numpy(dtype="float64")).sum()
print(f"\n[최종 검증] Feature NaN: {nan_cnt} | inf: {inf_cnt}")

# --- 5) 저장 컬럼 구성 (메타 10 + Feature 14) ---
META_COLS = ["source_segment_id","window_id","window_start","window_end","win_n",
             "current_tool","split","anomaly_type","stage","label"]
save_cols = META_COLS + FEATURE_COLS
df_save = df_final[save_cols].copy()

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_save.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
size_mb = OUT_PATH.stat().st_size / (1024 * 1024)
print(f"\n[저장 완료] {OUT_PATH.resolve()}")
print(f"파일 크기: {size_mb:.2f} MB | 저장 컬럼 수: {len(save_cols)}")

# --- 6) 재로드 검증 ---
reloaded = pd.read_csv(OUT_PATH, encoding="utf-8-sig", low_memory=False)
print("\n" + "=" * 60)
print("[재로드 검증]")
print("=" * 60)
print("shape:", reloaded.shape)
print("저장 행수 == 재로드 행수:", len(df_save) == len(reloaded), f"({len(reloaded):,})")

rl_nan = reloaded[FEATURE_COLS].isna().sum().sum()
rl_inf = np.isinf(reloaded[FEATURE_COLS].to_numpy(dtype="float64")).sum()
print(f"재로드 NaN: {rl_nan} | inf: {rl_inf}")

print("\n[label 분포]")
print(reloaded["label"].value_counts().sort_index().to_string())
print("\n[split 분포]")
print(reloaded["split"].value_counts().reindex(["train","validation","test"]).to_string())

tools = sorted(reloaded["current_tool"].unique())
print("\n[포함된 공구 목록]:", tools)
print("공구 15 제거됨:", 15 not in tools, "| 공구 16 제거됨:", 16 not in tools)
print("포함 공구 수:", len(tools))

print("\n[안내] training_dataset.csv 저장·검증 완료.")

[제외 대상] 상대 Feature NaN Window: 60개
사유: Train 기간에 해당 공구의 정상(stage=normal) Window가 없어
      공구별 정상 baseline을 만들 수 없음 → 상대 Feature 계산 불가

[제외 Window: 공구 × split 별 개수]
current_tool  split     
15            validation    30
16            validation    30

제외 전 Window: 4,620  →  제외 후 Window: 4,560

[최종 검증] Feature NaN: 0 | inf: 0

[저장 완료] D:\kc_project\예지보전\MCT-Data-Preprocessing\output\training_dataset.csv
파일 크기: 1.02 MB | 저장 컬럼 수: 24

[재로드 검증]
shape: (4560, 24)
저장 행수 == 재로드 행수: True (4,560)
재로드 NaN: 0 | inf: 0

[label 분포]
label
0    1368
1    3192

[split 분포]
split
train         3630
validation     370
test           560

[포함된 공구 목록]: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(8), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(18), np.int64(19)]
공구 15 제거됨: True | 공구 16 제거됨: True
포함 공구 수: 14

[안내] training_dataset.csv 저장·검증 완료.
